# 08 - SPM blending on GLORIA: Cordeiro (2022) OWT vs invRrs water types

GRSl2bgen blends four SPM algorithms with the Cordeiro (2022) OWT (`SPM_OWTblend_Ta2025`): the two closest classes
(Euclidean distance on Rrs, 443-783 nm) weighted by the inverse distance, each class with its algorithm and validity
range (1: Jiang et al. 2021 green, 0-50; 2: Jiang red, 10-500; 3: Zhang et al. 2014 705 nm, 20-1000; 4: Binding
et al. 2010 740 nm, 50-2000 g m$^{-3}$). This notebook tests whether the invRrs water types of 03 (14 types,
oligotrophic to extremely turbid) do better, on the **GLORIA** in situ data set (Lehmann et al. 2023): above-water
hyperspectral Rrs with laboratory TSS, band-averaged on the MSI bands (S2A).

Blendings compared (same code path as the processor: `GRSl2bgen.OWT` and the algorithms of `GRSl2bgen.Spm`):

- `Cordeiro (GRSl2bgen)`: the product of GRSl2bgen up to 1.2.0 (Cordeiro OWT, Euclidean distance);
- `invRrs WT (GRSl2bgen)`: the product that replaces it, computed with the processor code (`OWT_process`, `Spm`):
  invRrs water types classified by spectral angle, 2 classes, recipe `SPM_OWT_RECIPE` of
  `GRSl2bgen.suspended_particulate_matter` and fallback to the best class without validity range;
- `invRrs WT, Cordeiro algorithms`: the new types, each mapped to the algorithm of its expected Cordeiro class;
- `invRrs WT, best algorithm (CV)`: for each new type, the algorithm with the lowest error on the other half of the
  data sets (2-fold cross-validation by data set, so that a recipe is never scored on the samples that chose it);
- single algorithms, for reference.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt

from ta_common import *

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'font.size': 10})

In [ ]:
from GRSl2bgen.owt import OWT
from GRSl2bgen.suspended_particulate_matter import Spm

GLORIA = Path('/DATA/projet/vrac/gloria/GLORIA_2022')
WT_LIB = load_owt(WT_OWT_FILE)
CORD = load_owt(CORDEIRO_FILE)
WT_CODES = list(WT_LIB.owt.values)
# expected Cordeiro class of each invRrs water type (03)
CORDEIRO_OF = {'WT01': 1, 'WT02': 1, 'WT03': 1, 'WT04': 2, 'WT05': 2, 'WT06': 2, 'WT07': 2, 'WT08': 1,
               'WT09': 2, 'WT10': 1, 'WT11': 3, 'WT12': 3, 'WT13': 4, 'WT14': 4}
N_CLASSES = 2

## 1. GLORIA: selection and MSI bands

In [ ]:
meta = pd.read_csv(GLORIA / 'GLORIA_meta_and_lab.csv', index_col=0, low_memory=False)
flags = pd.read_csv(GLORIA / 'GLORIA_qc_flags.csv', index_col=0).reindex(meta.index)
rrs = pd.read_csv(GLORIA / 'GLORIA_Rrs.csv', index_col=0).reindex(meta.index)
rrs.columns = rrs.columns.str.replace('Rrs_', '').astype(float)
steps = [('GLORIA records', pd.Series(True, index=meta.index)), ('TSS measured', meta.TSS > 0),
         ('no QC flag', flags.Flagged == 0), ('Rrs complete 400-800 nm', rrs.loc[:, 400:800].notna().all(axis=1))]
sel = pd.Series(True, index=meta.index)
for label, cond in steps:
    sel &= cond.fillna(False)
    print(f'{label:28s} {int(sel.sum()):5d}')
data = meta.loc[sel, ['Dataset_ID', 'Water_body_type', 'TSS', 'Chla', 'aCDOM440', 'Latitude', 'Longitude']].copy()
data['water_body'] = data.Water_body_type.map({1: 'lake', 2: 'estuary', 3: 'coastal', 4: 'river', 5: 'other'})
R = rrs.loc[sel]
WL = R.columns.values
s2 = s2_convolution(WL, R.values)                        # NaN where the band is not covered (842, 865 nm)
S2 = pd.DataFrame(s2, index=R.index, columns=BANDS)
raster = xr.Dataset({'Rrs': (('wl', 'y', 'x'), S2.T.values[:, None, :])},
                    coords={'wl': BANDS, 'y': [0], 'x': np.arange(len(S2))})
print(f'{len(data)} samples, {data.Dataset_ID.nunique()} data sets; TSS quantiles (g m-3):',
      data.TSS.quantile([0.05, 0.5, 0.95]).round(1).to_dict())
print('valid MSI bands:', S2.notna().mean().round(2).to_dict())

## 2. Classifications and SPM

In [ ]:
def owt(lib, name):
    lib = lib.drop_vars([c for c in lib.coords if c not in ('owt', 'wl')])
    o = OWT(raster, xowt=lib, owt_database_name=name, spectral_distance='euclidean', Nclasses_to_be_saved=N_CLASSES)
    return o.multi_process()


x_cord = owt(CORD, 'Cordeiro2022SPM')
x_wt = owt(WT_LIB, 'WT')
spm = Spm(raster)
spm.process()
est = pd.DataFrame(index=data.index)
# previous GRSl2bgen product (up to 1.2.0): Cordeiro OWT, Euclidean, its recipe, no fallback
CORD_RECIPE = {'jiang_green': ([1], (0, 50)), 'jiang_red': ([2], (10, 500)), 'zhang2014': ([3], (20, 1000)),
               'binding2010': ([4], (50, 2000))}
est['Cordeiro (GRSl2bgen)'] = spm.spm_blending(x_cord, Nclasses=N_CLASSES, owt_dist_var='owt_dist_Cordeiro2022SPM',
                                               owt_index_var='owt_index_Cordeiro2022SPM', recipe=CORD_RECIPE,
                                               fallback=False).values.ravel()
# current GRSl2bgen product: invRrs water types (spectral angle), processor code path
from GRSl2bgen.owt import OWT_process
op = OWT_process(raster, chunk=None)
op.execute()
spm_new = Spm(raster, xowt_prod=op.xowt_invRrsWT)
spm_new.process()
est['invRrs WT (GRSl2bgen)'] = spm_new.output['SPM_OWTblend_Ta2025'].values.ravel()
est['Nechad 665'] = spm.output['SPM_nechad'].values.ravel()
est['OBS2CO 665/865'] = spm.output['SPM_obs2co'].values.ravel()

b = {w: raster.Rrs.sel(wl=w) for w in BANDS}
ALGOS = {  # algorithm: (function of the bands, validity range g m-3), as in Spm.spm_blending
    'Jiang green': (lambda: Spm._spm_jiang_green(b[443], b[490], b[560], b[665]), (0, 50)),
    'Jiang red': (lambda: Spm._spm_jiang_red(b[443], b[490], b[560], b[665]), (10, 500)),
    'Zhang 705': (lambda: Spm._spm_zhang2014(b[705]), (20, 1000)),
    'Binding 740': (lambda: Spm._spm_binding2010(b[740]), (50, 2000)),
    'Nechad 665': (lambda: spm.spm_N10(valid_limit=[0, 2000]), (0, 2000)),
}
A = {k: np.asarray(f().values, float).ravel() for k, (f, _) in ALGOS.items()}
RANGE = {k: r for k, (_, r) in ALGOS.items()}
CORD_ALGO = {1: 'Jiang green', 2: 'Jiang red', 3: 'Zhang 705', 4: 'Binding 740'}
for k in ('Jiang green', 'Jiang red', 'Zhang 705', 'Binding 740'):
    est[k] = A[k]

idx_c = x_cord['owt_index_Cordeiro2022SPM'].values[:, 0, :]
idx_w = x_wt['owt_index_WT'].values[:, 0, :]
dist_w = x_wt['owt_dist_WT'].values[:, 0, :]
data['cordeiro_class'] = idx_c[0]
data['wt_class'] = [WT_CODES[int(i) - 1] if np.isfinite(i) else None for i in idx_w[0]]
pd.crosstab(data.wt_class, data.cordeiro_class.astype('Int64'), margins=True)

In [ ]:
def blend(idx, dist, recipe, sub=slice(None), limits=True, eps=1e-12):
    # inverse-distance blending of the N best classes, as Spm.spm_blending; recipe: {1-based class: algorithm}
    num = np.zeros(idx.shape[1]); den = np.zeros(idx.shape[1])
    for k in range(idx.shape[0]):
        w = 1 / (dist[k] + eps)
        for cls, algo in recipe.items():
            e = A[algo][sub]
            lo, hi = RANGE[algo]
            v = (idx[k] == cls) & np.isfinite(e) & np.isfinite(w)
            if limits:
                v &= (e >= lo) & (e <= hi)
            num[v] += e[v] * w[v]; den[v] += w[v]
    return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)


recipe_wt_cord = {i + 1: CORD_ALGO[CORDEIRO_OF[c]] for i, c in enumerate(WT_CODES)}
est['invRrs WT, Cordeiro algorithms'] = blend(idx_w, dist_w, recipe_wt_cord)


def mdsa(e, o):
    q = np.log(e / o); q = q[np.isfinite(q)]
    return 100 * (np.exp(np.median(np.abs(q))) - 1) if len(q) else np.nan


def sspb(e, o):
    q = np.log(e / o); q = q[np.isfinite(q)]
    m = np.median(q) if len(q) else np.nan
    return 100 * np.sign(m) * (np.exp(np.abs(m)) - 1)


# 2-fold cross-validation by data set: the best algorithm of each new type is chosen on one half of the data sets
# (lowest MdSA, at least MIN_N valid estimates, else the algorithm of its expected Cordeiro class) and applied to the other
MIN_N = 15
ds = data.Dataset_ID.astype(str).values
fold = (pd.util.hash_array(ds.astype(object)) % 2).astype(int)
cv = np.full(len(data), np.nan)
chosen = {}
for f in (0, 1):
    train, test = fold != f, fold == f
    rec = {}
    for i, c in enumerate(WT_CODES):
        m = train & (idx_w[0] == i + 1)
        scores = {}
        for algo in ALGOS:
            e, o = A[algo][m], data.TSS.values[m]
            lo, hi = RANGE[algo]
            ok = np.isfinite(e) & (e >= lo) & (e <= hi)
            if ok.sum() >= MIN_N:
                scores[algo] = mdsa(e[ok], o[ok]) + 100 * (1 - ok.mean())    # penalise the invalid estimates
        rec[i + 1] = min(scores, key=scores.get) if scores else CORD_ALGO[CORDEIRO_OF[c]]
    chosen[f] = rec
    cv[test] = blend(idx_w[:, test], dist_w[:, test], rec, sub=test)
est['invRrs WT, best algorithm (CV)'] = cv
pd.DataFrame({f'fold {f}': {WT_CODES[k - 1]: v for k, v in r.items()} for f, r in chosen.items()})

## 3. Scores

MdSA (median symmetric accuracy, %) and SSPB (symmetric signed percentage bias, %) of Morley et al. (2018) on the
valid estimates, and the share of samples with a valid estimate (an estimate outside the validity ranges of all the
blended classes is NaN, as in the processor).

In [ ]:
obs = data.TSS.values


def scores(mask=slice(None)):
    rows = {}
    for k in est.columns:
        e, o = est[k].values[mask], obs[mask]
        v = np.isfinite(e) & (e > 0)
        rows[k] = dict(N=int(v.sum()), valid=v.mean(), MdSA=mdsa(e[v], o[v]), SSPB=sspb(e[v], o[v]))
    return pd.DataFrame(rows).T


tab = scores()
tab.to_csv(DATA / 'spm_gloria_scores.csv')
tab.round(2).sort_values('MdSA')

In [ ]:
bins = pd.cut(data.TSS, [0, 5, 20, 100, np.inf], labels=['<5', '5-20', '20-100', '>100'])
by_bin = pd.concat({b_: scores((bins == b_).values)[['MdSA', 'valid']] for b_ in bins.cat.categories}, axis=1)
by_wb = pd.concat({w: scores((data.water_body == w).values)[['MdSA', 'valid']] for w in ['lake', 'coastal', 'estuary', 'river']}, axis=1)
display(by_bin.round(2))
by_wb.round(2)

In [ ]:
show = ['Cordeiro (GRSl2bgen)', 'invRrs WT (GRSl2bgen)', 'invRrs WT, best algorithm (CV)', 'Nechad 665']
fig, axs = plt.subplots(1, len(show), figsize=(4.6 * len(show), 4.6), sharex=True, sharey=True)
colors = {'lake': 'tab:green', 'coastal': 'tab:blue', 'estuary': 'tab:purple', 'river': 'tab:brown', 'other': '0.5'}
for ax, k in zip(axs, show):
    for wb, g in data.groupby('water_body'):
        ax.scatter(g.TSS, est.loc[g.index, k], s=4, alpha=0.4, color=colors[wb], label=wb, rasterized=True)
    ax.plot([0.05, 5000], [0.05, 5000], 'k-', lw=0.8)
    for f in (0.5, 2):
        ax.plot([0.05, 5000], [0.05 * f, 5000 * f], 'k:', lw=0.6)
    t = tab.loc[k]
    ax.set(xscale='log', yscale='log', xlim=(0.05, 5000), ylim=(0.05, 5000), xlabel='in situ TSS (g m$^{-3}$)',
           title=f'{k}\nMdSA {t.MdSA:.0f} %, bias {t.SSPB:+.0f} %, valid {t.valid:.0%}')
axs[0].set_ylabel('SPM from GLORIA Rrs (g m$^{-3}$)')
axs[0].legend(markerscale=4, fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIG / 'spm_gloria_scatter.png')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
by_bin.xs('MdSA', axis=1, level=1).loc[show].T.plot.bar(ax=ax, rot=0)
ax.set(xlabel='in situ TSS (g m$^{-3}$)', ylabel='MdSA (%)', title='SPM error by TSS range (GLORIA, MSI bands)')
ax.legend(fontsize=8, frameon=False)
fig.savefig(FIG / 'spm_gloria_mdsa_by_tss.png')

## 4. Recipe proposed for GRSl2bgen

Best algorithm of each invRrs water type on all the GLORIA samples (to be used only if the cross-validated score
above beats the Cordeiro blending), with the number of samples of each type.

In [ ]:
final = {}
for i, c in enumerate(WT_CODES):
    m = idx_w[0] == i + 1
    sc = {}
    for algo in ALGOS:
        e, o = A[algo][m], obs[m]
        lo, hi = RANGE[algo]
        ok = np.isfinite(e) & (e >= lo) & (e <= hi)
        if ok.sum() >= MIN_N:
            sc[algo] = mdsa(e[ok], o[ok]) + 100 * (1 - ok.mean())
    final[c] = dict(n=int(m.sum()), algorithm=min(sc, key=sc.get) if sc else CORD_ALGO[CORDEIRO_OF[c]],
                    score=round(min(sc.values()), 1) if sc else None, cordeiro_algorithm=CORD_ALGO[CORDEIRO_OF[c]])
recipe_tab = pd.DataFrame(final).T
save_json({'scores': tab.round(3).to_dict(orient='index'), 'recipe': final, 'n_samples': len(data)},
          DATA / 'spm_gloria_recipe.json')
recipe_tab